# Урок 32. Итоговый проект: модель и метрики

**Дата:** 29.04.2027 · **60 минут** · 8–9 класс  
**Модуль 6.** Итоговый проект (уроки 30–36)  
**Опирается на:** урок 31 (данные проекта), уроки 19–23 (модели, baseline, метрики, переобучение), уроки 26–27 (картинки и бот).

Данные готовы — строим решение. Главное правило урока: **сначала baseline, потом модель, потом честная проверка**. Простая модель, которую ты понимаешь и честно проверил, лучше сложной, которую не можешь объяснить.

## Цели урока

1. Выбрать метод под тип своего проекта.
2. Посчитать baseline и обучить 1–2 модели.
3. Выбрать настройки по validation и один раз проверить на test.
4. Разобрать ошибки модели.
5. Заполнить журнал модели.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–5 | Выбор метода |
| 5–45 | Работа по шаблону своего типа проекта |
| 45–55 | Разбор ошибок, журнал модели |
| 55–60 | Короткий отчёт: какая метрика получилась |

## 1. Выбор метода

| Тип проекта | Baseline | Модели из курса | Главные метрики |
|-------------|----------|-----------------|-----------------|
| Таблица → классификация | `DummyClassifier(strategy="most_frequent")` | дерево решений, kNN (с масштабированием) | accuracy, матрица ошибок, precision, recall |
| Таблица → регрессия | `DummyRegressor()` (среднее) | линейная регрессия, дерево для регрессии | MAE |
| Картинки | угадывание самого частого класса | Teachable Machine или kNN на пикселях | accuracy на **отдельных** фото, матрица ошибок |
| Текст / бот | бот на правилах | мешок слов + kNN, порог «не понял» | доля верных намерений на тестовых фразах |

**Как выбирать между моделями:** на validation (урок 23). При равном качестве бери **более простую и понятную**.

## 2. Шаблон: табличная классификация

Ячейка ниже создаёт **пример** данных (те же столбцы, что в уроке 31, но 150 строк). Замени её на чтение своего `project_clean.csv`.

In [ ]:
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# ПРИМЕР ДАННЫХ — замени на: df = pd.read_csv("project_clean.csv")
random.seed(32)
rows = []
for i in range(150):
    days = random.randint(1, 14)
    done = random.randint(0, 100)
    lessons = random.randint(5, 8)
    clubs = random.randint(0, 4)
    chance = 0.1 + 0.05 * days + 0.006 * done - 0.08 * clubs - 0.05 * (lessons - 5)
    rows.append({
        "дней_до_дедлайна": days,
        "сделано_процентов": done,
        "уроков_в_день": lessons,
        "кружков": clubs,
        "успел": random.random() < chance,
    })
df = pd.DataFrame(rows)
print(df.shape)
print(df["успел"].value_counts())

### Шаг 1. X, y и разбиение на три части

Если данных меньше 100 строк, validation можно не выделять: выбери простую модель заранее (например, дерево глубины 2–3) и проверь её на test один раз.

In [ ]:
features = ["дней_до_дедлайна", "сделано_процентов", "уроков_в_день", "кружков"]
target = "успел"
X = df[features]
y = df[target]

X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=42, stratify=y_rest)
print("train:", len(X_train), "| validation:", len(X_val), "| test:", len(X_test))

### Шаг 2. Baseline

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
print("Baseline accuracy на validation:", round(accuracy_score(y_val, dummy.predict(X_val)), 2))

### Шаг 3. Модели и выбор настроек по validation

Положительный класс в нашем примере — «не успел» (его мы хотим поймать заранее). Поэтому для recall сравниваем с `False`: `pos_label=False`.

In [ ]:
def report(name, model, X_part, y_part):
    """Напечатать метрики модели на выбранной части данных."""
    pred = model.predict(X_part)
    acc = accuracy_score(y_part, pred)
    rec = recall_score(y_part, pred, pos_label=False)
    print(f"{name:22} accuracy {acc:.2f} | recall «не успел» {rec:.2f}")

for depth in [1, 2, 3, 5]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
    tree.fit(X_train, y_train)
    report(f"дерево глубины {depth}", tree, X_val, y_val)

max_train = X_train.max()
for k in [3, 7, 15]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train / max_train, y_train)
    report(f"kNN, k = {k}", knn, X_val / max_train, y_val)

`pos_label=False` — параметр, который говорит функции, какой класс считать положительным. По умолчанию положительный — `True`.

**Выбери** модель по главной метрике на validation и **запиши** выбор в журнал модели. В примере главная метрика — recall «не успел»: лучше всех на validation дерево глубины 1, к тому же оно самое простое. Его и проверим на test.

### Шаг 4. Итоговая проверка на test — один раз

In [ ]:
final = DecisionTreeClassifier(max_depth=1, random_state=0)   # ← твоя выбранная модель
final.fit(X_train, y_train)

report("baseline на test", dummy, X_test, y_test)
report("модель на test", final, X_test, y_test)
report("модель на train", final, X_train, y_train)
print(confusion_matrix(y_test, final.predict(X_test)))

На test recall получился ниже, чем на validation (0.79 против 0.93). Это нормально: validation маленькая, и выбор по ней немного «подстроился» под неё. Именно поэтому в отчёте пишут результат **на test** — и сравнивают его с baseline (там recall 0).

### Шаг 5. Объяснение модели и разбор ошибок

У дерева можно показать правила — это сильный аргумент на защите. Разбор ошибок показывает, **где** модель слаба.

In [ ]:
print(export_text(final, feature_names=features))

errors = X_test.copy()
errors["правда"] = y_test
errors["прогноз"] = final.predict(X_test)
errors[errors["правда"] != errors["прогноз"]]

## 3. Шаблон: табличная регрессия

Если твоя цель — число, используй этот шаблон (подставь свои данные и столбцы).

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# ПРИМЕР: предсказываем, сколько процентов работы будет сделано к дедлайну
X_r = df[["дней_до_дедлайна", "уроков_в_день", "кружков"]]
y_r = df["сделано_процентов"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_r, y_r, test_size=0.25, random_state=42)

base = DummyRegressor().fit(Xr_train, yr_train)
lin = LinearRegression().fit(Xr_train, yr_train)
print("MAE baseline:", round(mean_absolute_error(yr_test, base.predict(Xr_test)), 1))
print("MAE линейной регрессии:", round(mean_absolute_error(yr_test, lin.predict(Xr_test)), 1))
print(pd.Series(lin.coef_, index=X_r.columns).round(2))

В этом примере линейная регрессия **не лучше** baseline: мы генерировали «сделано процентов» случайно, связи с признаками нет. Это тоже честный результат — и о нём нужно так и сказать на защите: «модель не нашла закономерности; возможно, нужны другие признаки».

## 4. Бот: чек-лист готовности

Используй код урока 27 (`normalize`, `CountVectorizer`, `KNeighborsClassifier(metric="cosine")`, `safe_intent`).

- [ ] есть бот на правилах — это baseline;
- [ ] есть ML-бот, обученный на обучающих фразах;
- [ ] есть **отдельные** тестовые фразы (не из обучения, лучше написанные другими людьми);
- [ ] посчитана доля верных намерений для обоих ботов;
- [ ] работает ответ «не понял» с порогом;
- [ ] есть таблица ошибок: фраза → что ответил → почему ошибся.

## 5. Картинки: чек-лист готовности

- [ ] модель обучена (Teachable Machine или kNN на пикселях, урок 26);
- [ ] проверка на **отдельных** фото, которых не было в обучении: сколько верно по каждому классу;
- [ ] заполнена матрица ошибок (можно вручную: правда × ответ);
- [ ] проверены сложные условия: другой свет, фон, расстояние;
- [ ] записано, где модель ошибается чаще всего.

## 6. Если метрика плохая — чек-лист отладки

1. Хватает ли данных? (урок 31)
2. Нет ли ошибки в целевом столбце (перепутаны классы, текст вместо `True / False`)?
3. Признаки вообще связаны с целью? Вернись к разведке (урок 12).
4. Для kNN — масштабирование по train сделано? (урок 20)
5. Не переобучение ли? Сравни train и validation (урок 23).
6. Может, задача слишком сложная — сократи число классов.

**Подозрительно хорошая метрика** (100% или почти) — первым делом ищи **утечку** (урок 17).

## 7. Журнал модели

| Пункт | Мой ответ |
|-------|-----------|
| Признаки X / классы / намерения | |
| Разбиение (сколько в train, validation, test; `random_state`) | |
| Baseline и его метрика | |
| Какие модели и настройки пробовал (на validation) | |
| Что выбрал и почему | |
| Итоговые метрики на test | |
| Сравнение train и test (есть ли переобучение) | |
| Где модель ошибается | |
| Что улучшил бы, будь больше времени | |

## Работа с ИИ-помощником

- «Моя модель: accuracy на train 1.0, на validation 0.7. Что проверить в первую очередь?»
- «Объясни эти правила дерева простыми словами: …» — потом проверь объяснение на примерах.
- Не принимай совет «просто возьми модель посложнее» без проверки на validation (урок 29).

## Домашнее задание

1. Доведи модель до итоговой проверки на test.
2. Заполни журнал модели.
3. Подумай, как будет выглядеть демо: что ты покажешь зрителям за 2–3 минуты?

## Самопроверка

1. Какой у тебя baseline и насколько модель лучше него?
2. По какой выборке ты выбирал настройки? Сколько раз смотрел на test?
3. Есть ли у модели переобучение?
4. На каких примерах модель ошибается и почему?

**Следующий урок (06.05.2027):** проект — демо, карточка ограничений, подготовка выступления.